# 20261113 · 课程串讲：Agent 工程化 —— create_agent · 中间件 · MCP · Deep Agents

上一讲（1111）把 RAG 检索链路做扎实了；这一讲把「检索封成工具」之后的事补上：**怎么用 LangChain 的 `create_agent` 搭一个能循环调工具的 Agent、怎么用中间件控制它、怎么用 MCP 把工具做成通用件、怎么用 Deep Agents 把长任务撑住。**

课堂三份材料：`langgraph_create_agent.ipynb`（create_agent + 中间件）、`MCP.ipynb`（协议 + 自建 Server）、`DeepAgents课件.md`（六讲 harness + Skills）。

| 讲 | 主题 | 关键结论 | 本讲对应节 |
|---|---|---|---|
| 1 | `create_agent` 与循环 | 模型要 `tool_calls` → 执行工具 → `ToolMessage` 回填 → 再问模型 | 第 1 节 |
| 2 | 结构化输出 | Prompt / Function Calling / 推理层约束，可靠性递增 | 第 2 节 |
| 3 | 中间件 | `wrap_model_call` 管模型，`wrap_tool_call` 管工具 | 第 3、4 节 |
| 4 | 上下文治理 | 摘要压缩历史，动态提示按状态改系统提示 | 第 5 节 |
| 5 | MCP | JSON-RPC 2.0 + SSE/Stdio，工具做成「USB 设备」 | 第 6 节 |
| 6 | Deep Agents | 文件系统 / 子代理 / 权限 / 人在回路 / Skills | 第 7、8、9 节 |

## 执行说明

- 环境：`week23` 的 Python 3.11 虚拟环境（`week23-langchain` 内核），本讲新增依赖 `deepagents`、`langchain-mcp-adapters`、`mcp`。
- 模型：沿用 `week23/.env` 的 `LLM_PROVIDER=qwen`（`qwen3.8-flash`）。课堂脚本里的 `http://192.168.1.105:11434` 是教室 Ollama，本讲一律换成云端配置，本机不需要起 Ollama。
- 开关：`RUN_LIVE` 控制所有联网单元（云端对话）。默认关闭，离线仍可读完全文；下文保留的输出来自一次 `RUN_LIVE=True` 的实际运行。
- 课堂文件对照：`test1.py`（回调）、`test2.py`（结构化输出）、`test3~5.py`（中间件）、`test_deepagnt/`（Skills 会议纪要 demo）。
- 唯一的外部文件是 `20261113_lab/mcp_library_server.py`（自建 MCP Server），其余沙箱目录由本 notebook 自己生成。

## 0. 环境自检

| 组件 | 用途 | 本讲取值 |
|---|---|---|
| 千问兼容接口 | 全部 LLM 调用 | `week23/.env` 的 `QWEN_BASE_URL` |
| `deepagents` | Deep Agents harness | 0.7.38 |
| `langchain-mcp-adapters` | 把 MCP 工具转成 LangChain 工具 | 0.3.2 |
| `mcp` | 自建 MCP Server（`FastMCP`） | 1.30.0 |

课堂笔记里写的是 `from mcp.server.mcpserver import MCPServer`；本机装的 `mcp 1.30.0` 里类名是 `mcp.server.FastMCP`，写法按装到的版本走。

In [1]:
import importlib.metadata as md
import json
import shutil
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

RUN_LIVE = False  # 云端对话开关：False 时只做离线演示（JSON-RPC、目录生成等）

LAB_CONFIG_DIR = Path(r"D:\project\step1\week23\20261027")
NOTEBOOK_DIR = Path.cwd()
LAB_DIR = NOTEBOOK_DIR / "20261113_lab"        # 自建 MCP Server 所在目录
SANDBOX = LAB_DIR / "sandbox"                  # FilesystemBackend 的根目录
SERVER_PY = LAB_DIR / "mcp_library_server.py"

print("== 包版本 ==")
for name in ("langchain", "langchain-core", "langchain-openai", "langgraph",
             "deepagents", "langchain-mcp-adapters", "mcp"):
    try:
        print(f"  {name:24s}{md.version(name)}")
    except md.PackageNotFoundError:
        print(f"  {name:24s}[未安装]")
print(f"\n自建 MCP Server：{SERVER_PY}（存在={SERVER_PY.exists()}）")
llm = None

== 包版本 ==
  langchain               1.5.0
  langchain-core          1.8.0
  langchain-openai        1.7.3
  langgraph               1.2.18
  deepagents              0.7.38
  langchain-mcp-adapters  0.3.2
  mcp                     1.30.0

自建 MCP Server：D:\project\step1\week25\20261113_lab\mcp_library_server.py（存在=True）


In [2]:
if RUN_LIVE:
    if str(LAB_CONFIG_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_CONFIG_DIR))
    import lab_config

    if lab_config.PROVIDER != "qwen":
        raise ValueError("本讲按 week23/.env 的 LLM_PROVIDER=qwen 编写，请先切换配置。")

    llm = lab_config.get_model(temperature=0)
    print(f"对话模型：{lab_config.MODEL} @ {lab_config.BASE_URL}")
    print("探测：", llm.invoke("请只回复 OK。").content.strip())
else:
    print("离线模式：不加载云端模型，第 1~9 节的联网单元格会打印跳过说明。")

对话模型：qwen3.8-flash @ https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1


探测： OK


## 1. `create_agent`：Agent 的最小骨架

`create_agent` 返回一张编译好的 `StateGraph`，输入输出都是 `{"messages": [...]}`。它的循环只有一句话：

```text
模型返回带 tool_calls 的 AIMessage
        ↓
执行工具，把结果包成 ToolMessage 追加进 messages
        ↓
再调用模型（此时它能看见工具结果）
        ↓
没有 tool_calls → 结束，返回完整消息列表
```

课堂 `test1.py` 用手写回调演示了同一个结构：`test(a, b, hook)` 里判断有没有 `hook`，有就交给 `hook` 处理 —— 中间件就是 Agent 世界的 `hook`。

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def check_weather(location: str) -> str:
    """返回指定位置的天气预报。"""
    return f"{location} 晴，20℃，东南风 2 级"


if RUN_LIVE:
    agent = create_agent(
        model=llm,
        tools=[check_weather],
        system_prompt="你是天气助手，回答前先调用工具。",
    )
    result = agent.invoke({"messages": [("user", "北京天气如何？")]})

    for msg in result["messages"]:
        calls = getattr(msg, "tool_calls", None) or []
        if calls:
            print(f"{type(msg).__name__:16s} tool_calls={[c['name'] for c in calls]}")
        else:
            print(f"{type(msg).__name__:16s} {str(msg.content)[:60]}")
    print("\n最终回答：", result["messages"][-1].content)
else:
    print("离线模式：跳过第 1 节调用。")

HumanMessage     北京天气如何？
AIMessage        tool_calls=['check_weather']
ToolMessage      北京 晴，20℃，东南风 2 级
AIMessage        北京当前天气：晴，气温 20℃，东南风 2 级。

最终回答： 北京当前天气：晴，气温 20℃，东南风 2 级。


### `create_agent` 参数速查（只留本讲用得到的）

| 参数 | 作用 | 本讲用在哪 |
|---|---|---|
| `model` | 模型实例或 `provider:model` 字符串 | 全部 |
| `tools` | 工具列表；`None` 时图里没有工具循环 | 第 1、2、4 节 |
| `system_prompt` | 系统提示，可被 `dynamic_prompt` 中间件覆盖 | 第 1、5 节 |
| `middleware` | 中间件序列，按顺序包裹模型 / 工具 | 第 3~5 节 |
| `response_format` | `ToolStrategy` / `ProviderStrategy` / Pydantic 类 | 第 2 节 |
| `checkpointer` | 线程内记忆（对话历史） | 第 5 节 |
| `store` | 跨线程持久化 | 第 5 节 |
| `context_schema` | 运行时上下文的类型（用户身份等） | 第 4 节 |

> 课堂笔记里还列了 `state_schema`、`interrupt_before/after`、`cache`、`debug`，属于「知道有」的层级。

## 2. 结构化输出：三档可靠性

| 档位 | 做法 | 原理 | 依赖 |
|---|---|---|---|
| ① Prompt + 解析器 | `JsonOutputParser().get_format_instructions()` 塞进提示词 | 靠模型自觉 | 什么都行，最不可靠 |
| ② Function Calling | `ToolStrategy(模型类)` / `with_structured_output(method="function_calling")` | 把 schema 伪装成一个工具，模型「调用」它 | 模型支持工具调用 |
| ③ 推理层约束 | `ProviderStrategy(模型类)` / `response_format={"type":"json_schema"}` | 采样时按 JSON Schema 约束 token | **厂商必须支持** |

`create_agent(response_format=...)` 走的是 ②③：结果不塞进 `messages`，而是放在返回 state 的 `structured_response` 里。

In [4]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy


class Weather(BaseModel):
    weather: str = Field(description="天气状况")
    temperature: int = Field(description="温度（摄氏度）")


if RUN_LIVE:
    # 档位 ②：ToolStrategy —— 把输出 schema 包装成一个工具，靠 function calling 保证格式
    agent_tool = create_agent(
        model=llm,
        tools=[check_weather],
        response_format=ToolStrategy(Weather),
    )
    res_tool = agent_tool.invoke({"messages": [("user", "北京天气怎么样？")]})
    structured = res_tool.get("structured_response")
    print("ToolStrategy：", structured, type(structured).__name__)
    print("按字段用：", structured.weather, structured.temperature)
else:
    print("离线模式：跳过。")

ToolStrategy： weather='晴' temperature=20 Weather
按字段用： 晴 20


In [5]:
if RUN_LIVE:
    # 档位 ③：ProviderStrategy —— 走模型原生的 response_format=json_schema
    # 能否成功取决于服务商与模型是否真的支持；不支持会在解析阶段抛 StructuredOutputValidationError
    try:
        agent_native = create_agent(
            model=llm,
            tools=[check_weather],
            response_format=ProviderStrategy(Weather),
        )
        res_native = agent_native.invoke({"messages": [("user", "北京天气怎么样？")]})
        print("ProviderStrategy：", res_native.get("structured_response"))
    except Exception as exc:  # noqa: BLE001
        print("ProviderStrategy 失败：", type(exc).__name__)
        print(str(exc)[:180])
        print("\n结论：③ 只有模型侧真的实现了 json_schema 约束才可用；")
        print("      同一个接口换个模型（如 qwen3-max）可能就好，属于「厂商能力」而非「代码写法」问题。")
        print("      工程上常用 ② 打底 + ③ 提速，别把 ③ 当默认。")
else:
    print("离线模式：跳过。")

ProviderStrategy 失败： StructuredOutputValidationError
Failed to parse structured output for tool 'Weather': Native structured output expected valid JSON for Weather, but parsing failed: Expecting value: line 1 column 1 (char 0)..

结论：③ 只有模型侧真的实现了 json_schema 约束才可用；
      同一个接口换个模型（如 qwen3-max）可能就好，属于「厂商能力」而非「代码写法」问题。
      工程上常用 ② 打底 + ③ 提速，别把 ③ 当默认。


## 3. `@wrap_model_call`：包住模型调用

`handler(request)` 才是「真正去问模型」的那一步。你在它前后能做的事：

| 用法 | 做法 | 课堂文件 |
|---|---|---|
| 重试 | `for` 循环包 `handler(request)` | `test3.py` 注释段 |
| 降级 | 失败后 `request.override(model=备用模型)` 再调一次 | `test3.py` |
| 改写响应 | 返回新的 `ModelResponse`，或直接返回 `AIMessage`（会自动转换） | `test3.py` |
| 观测 | 打印 `len(request.state["messages"])` 等 | 本讲 |

下面用「主模型名字写错 → 自动降级」演示，失败是必然的，方便观察。

In [6]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_openai import ChatOpenAI


@wrap_model_call
def fallback_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """主模型挂了就换备用模型再问一次。"""
    try:
        return handler(request)
    except Exception as exc:  # noqa: BLE001
        print(f"[降级] 主模型失败：{type(exc).__name__}")
        return handler(request.override(model=llm))


if RUN_LIVE:
    import lab_config  # 第 0 节已把配置目录加入 sys.path

    broken = ChatOpenAI(
        model="not-a-real-model",
        base_url=lab_config.BASE_URL,
        api_key=lab_config.API_KEY,
    )
    agent_fb = create_agent(model=broken, tools=[], middleware=[fallback_model])
    res_fb = agent_fb.invoke({"messages": [("user", "用一句话说明什么是 Agent 的中间件。")]})
    print("回答：", res_fb["messages"][-1].content[:120])
else:
    print("离线模式：跳过。")

[降级] 主模型失败：OpenAIModelNotFoundError


回答： Agent 中间件是位于大语言模型与外部工具或环境之间的抽象层，负责处理请求路由、状态管理、错误重试及上下文增强等逻辑，从而简化 Agent 的构建并提升其可靠性与可维护性。


In [7]:
@wrap_model_call
def trace_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """观测：记录每次模型调用前的消息数、返回的消息类型。"""
    print(f"→ 调用模型，当前消息 {len(request.state['messages'])} 条")
    response = handler(request)
    print(f"← 返回 {type(response.result[0]).__name__}")
    return response


if RUN_LIVE:
    agent_trace = create_agent(model=llm, tools=[check_weather], middleware=[trace_model])
    agent_trace.invoke({"messages": [("user", "上海天气如何？")]})
    print("\n注意：带工具调用时这个中间件会被调用两次（一次要工具，一次给答案）。")
else:
    print("离线模式：跳过。")

→ 调用模型，当前消息 1 条


← 返回 AIMessage
→ 调用模型，当前消息 3 条


← 返回 AIMessage

注意：带工具调用时这个中间件会被调用两次（一次要工具，一次给答案）。


## 4. `@wrap_tool_call`：包住工具调用

`handler(request)` 执行工具并返回 `ToolMessage`。四个典型用途：日志计时、失败重试、**权限拦截**、结果缓存。

拦截时必须自己构造 `ToolMessage`，并且 **`tool_call_id` 不能省** —— 模型一条 `AIMessage` 里可能有好几个 `tool_calls`，全靠这个 id 对上号。

In [8]:
from dataclasses import dataclass
from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import ToolMessage


@dataclass
class User:
    user_id: str


BLOCKED_TOOLS = {"user1": {"check_weather"}}

@wrap_tool_call
def check_permission(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    user_id = request.runtime.context.user_id
    tool_name = request.tool_call["name"]
    if tool_name in BLOCKED_TOOLS.get(user_id, set()):
        print(f"[拦截] {user_id} 无权调用 {tool_name}")
        return ToolMessage(
            content=f"用户 {user_id} 无权限调用工具 {tool_name}，请直接告知用户。",
            tool_call_id=request.tool_call["id"],
        )
    return handler(request)


if RUN_LIVE:
    guarded = create_agent(
        model=llm,
        tools=[check_weather],
        middleware=[check_permission],
        context_schema=User,
    )
    res_g = guarded.invoke(
        {"messages": [("user", "北京天气如何？")]},
        context=User(user_id="user1"),
    )
    print("最终回答：", res_g["messages"][-1].content[:120])

    res_g2 = guarded.invoke(
        {"messages": [("user", "北京天气如何？")]},
        context=User(user_id="user2"),
    )
    print("换个用户：", res_g2["messages"][-1].content[:80])
else:
    print("离线模式：跳过。")

[拦截] user1 无权调用 check_weather


最终回答： 抱歉，当前账号没有权限调用天气查询工具，因此无法为您获取北京的实时天气预报。

建议您通过以下方式查询：
- 查看手机自带的天气应用
- 访问中国天气网（weather.com.cn）
- 在搜索引擎中直接搜索“北京天气”


换个用户： 北京当前天气为晴，气温20℃，东南风2级。


In [9]:
_tool_cache: dict[str, ToolMessage] = {}

@wrap_tool_call
def cache_tool_result(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """工具名 + 排序后的参数做 key，相同调用直接复用结果。"""
    key = f"{request.tool_call['name']}:{sorted(request.tool_call['args'].items())}"
    if key in _tool_cache:
        print(f"[缓存命中] {key}")
        return _tool_cache[key]
    print(f"[实际执行] {key}")
    response = handler(request)
    _tool_cache[key] = response
    return response


if RUN_LIVE:
    cached = create_agent(model=llm, tools=[check_weather], middleware=[cache_tool_result])
    for i in range(2):
        print(f"\n--- 第 {i + 1} 次提问 ---")
        cached.invoke({"messages": [("user", "广州天气如何？")]})
else:
    print("离线模式：跳过。")


--- 第 1 次提问 ---


[实际执行] check_weather:[('location', '广州')]



--- 第 2 次提问 ---


[缓存命中] check_weather:[('location', '广州')]


## 5. 上下文治理：摘要 + 动态提示

- `SummarizationMiddleware(model=小模型, trigger=("messages", N), keep=("messages", M))`：历史超过 N 条就压缩成摘要，只留最近 M 条原文。
- `@dynamic_prompt`：每次调用模型前重写 system prompt；`request.state` 看对话状态，`request.runtime.context` 看运行时上下文。

两者一个管「长度」，一个管「内容」，是长任务不爆上下文的两根支柱。

In [10]:
from langchain.agents.middleware import SummarizationMiddleware, dynamic_prompt

if RUN_LIVE:
    from langgraph.checkpoint.memory import MemorySaver

    sum_agent = create_agent(
        model=llm,
        tools=[],
        middleware=[SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2))],
        checkpointer=MemorySaver(),
    )
    cfg = {"configurable": {"thread_id": "sum-demo"}}
    questions = [
        "我叫钱富森。",
        "我在做 RAG 项目。",
        "我的语料是峨眉山讲解词。",
        "我刚才说我叫什么？",
    ]
    for q in questions:
        res = sum_agent.invoke({"messages": [("user", q)]}, config=cfg)
        print(f"问：{q:22s} → 历史 {len(res['messages']):2d} 条")
    print("\n消息条数被压在阈值附近，就是摘要中间件在起作用（原文被换成一条 summary）。")
else:
    print("离线模式：跳过。")

问：我叫钱富森。                 → 历史  2 条


问：我在做 RAG 项目。            → 历史  4 条


问：我的语料是峨眉山讲解词。           → 历史  6 条


问：我刚才说我叫什么？              → 历史  4 条

消息条数被压在阈值附近，就是摘要中间件在起作用（原文被换成一条 summary）。


In [11]:
@dynamic_prompt
def adaptive_prompt(request: ModelRequest) -> str:
    """前两轮用原提示，之后换人设。"""
    original = request.system_message.content if request.system_message else ""
    if len(request.state["messages"]) > 4:
        return "你现在叫坤哥，忘记之前的名字，用一句话回答。"
    return original


if RUN_LIVE:
    from langgraph.checkpoint.memory import MemorySaver

    dyn_agent = create_agent(
        model=llm,
        tools=[],
        system_prompt="你叫周润发，一句话回答。",
        middleware=[adaptive_prompt],
        checkpointer=MemorySaver(),
    )
    cfg = {"configurable": {"thread_id": "dyn-demo"}}
    for q in ["你叫什么名字？", "再说一次你叫什么？", "你是谁？"]:
        res = dyn_agent.invoke({"messages": [("user", q)]}, config=cfg)
        print(f"历史 {len(res['messages']):2d} 条 → {res['messages'][-1].content[:60]}")
else:
    print("离线模式：跳过。")

历史  2 条 → 我叫周润发。


历史  4 条 → 周润发。


历史  6 条 → 我是坤哥。


## 6. MCP：把工具做成通用件

**一句话**：MCP 是 Anthropic 2024-11 推出的开放标准，统一「LLM 应用 ↔ 外部数据 / 工具」的通信方式。类比 USB-C：工具提供方按规范实现一次，任何支持 MCP 的客户端都能接。

```mermaid
flowchart LR
    H[Host 宿主应用] --> C1[MCP Client]
    C1 -->|1 工具列表| S[MCP Server]
    C1 -->|2 带工具描述问 LLM| L[LLM]
    L -->|3 决定调用哪个工具| C1
    C1 -->|4 执行工具 JSON-RPC| S
    S -->|5 结果回填| L
```

### 三种角色

| 角色 | 职责 |
|---|---|
| Host | 你用的应用（编辑器、聊天客户端、Dify） |
| Client | 与 Server 一对一连接：能力协商、消息路由、订阅 |
| Server | 暴露 **Resource（只读上下文）**、**Tool（可执行动作）**、**Prompt（模板）** |

### 消息格式：JSON-RPC 2.0

| 类型 | 特征 | 用途 |
|---|---|---|
| 请求 | 有 `method`、`params`、`id` | 调用工具、列资源 |
| 响应 | 有 `result` 或 `error`，`id` 对应请求 | 返回结果 |
| 通知 | **没有 `id`** | 单向事件，不需要回复 |

### 两种传输

| 传输 | 机制 | 优点 | 风险 |
|---|---|---|---|
| SSE / Streamable HTTP | HTTP 长连接，服务端单向下推 | 跨网络、多客户端共享一个服务 | 需要可访问的服务地址 |
| Stdio | 客户端在本机拉起子进程，走标准输入输出 | 免部署、最稳 | **服务方代码跑在你机器上**，要审来源 |

课堂里 `MCP.ipynb` 用魔搭的 12306-mcp 演示 SSE，用 `npx -y 12306-mcp` 演示 Stdio。本讲改用**自建的图书馆 Server**，不依赖外网服务也能跑通全链路。

In [12]:
# JSON-RPC 2.0 三种消息：不需要任何服务，纯结构演示
request_msg = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
               "params": {"name": "check_stock", "arguments": {"book_name": "三体"}}}
response_msg = {"jsonrpc": "2.0", "id": 1,
                "result": {"content": [{"type": "text", "text": "《三体》可借数量：3"}]}}
notification_msg = {"jsonrpc": "2.0", "method": "notifications/progress", "params": {"progress": 50}}

for label, msg in [("请求", request_msg), ("响应", response_msg), ("通知", notification_msg)]:
    print(f"{label}：{json.dumps(msg, ensure_ascii=False)}")
print("\n注意：通知没有 id —— 这是「不需要回复」的唯一标志。")

请求：{"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "check_stock", "arguments": {"book_name": "三体"}}}
响应：{"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "《三体》可借数量：3"}]}}
通知：{"jsonrpc": "2.0", "method": "notifications/progress", "params": {"progress": 50}}

注意：通知没有 id —— 这是「不需要回复」的唯一标志。


### 自建 Server：Resource 与 Tool 的分工

`20261113_lab/mcp_library_server.py` 把同一个业务拆成两半：

- `@mcp.resource("library://rules")`、`@mcp.resource("student://{student_id}")`：**只读**的规则与档案，客户端主动读出来注入 system prompt；
- `@mcp.tool()` `check_stock`、`borrow_book`：**会改变状态**的动作，交给模型按需调用。

Resource 和 Tool 的区别不是技术限制，是**语义约定**：能被模型随手改的东西不该做成 Resource。

### 客户端一侧

`langchain_mcp_adapters` 提供两件事：

1. `await client.get_tools()` → 拿到可以直接塞给 `create_agent(tools=...)` 的 LangChain 工具；
2. `async with client.session("library") as session` → 直接读 Resource、列资源。

下面用 **stdio** 传输：客户端用 `sys.executable` 拉起 `mcp_library_server.py` 子进程，跑完自动退出。

> **Windows + Jupyter 的坑**：Jupyter 里 `sys.stderr` 是 ipykernel 的 `OutStream`，没有真实文件描述符；MCP 的 stdio 客户端默认把它当成子进程的 stderr，`subprocess` 会报 `UnsupportedOperation: fileno`。
> 解法是把 `errlog` 换成 `os.devnull`（下面几行补丁）。直接用 `python xxx.py` 跑同一个脚本时不需要。

In [13]:
import os
import mcp.client.stdio as mcp_stdio
import langchain_mcp_adapters.sessions as mcp_sessions
from langchain_mcp_adapters.client import MultiServerMCPClient

# ipykernel 的 sys.stderr 没有 fileno，换成 devnull 再交给子进程
_ERRLOG = open(os.devnull, "w", encoding="utf-8")
_orig_stdio_client = mcp_stdio.stdio_client
mcp_sessions.stdio_client = lambda server, errlog=None: _orig_stdio_client(
    server, errlog if errlog is not None else _ERRLOG
)

MCP_CONFIG = {
    "library": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(SERVER_PY)],
    }
}

client = MultiServerMCPClient(MCP_CONFIG)
mcp_tools = await client.get_tools()
print("工具：", [(t.name, t.description) for t in mcp_tools])

async with client.session("library") as session:
    rules = await session.read_resource("library://rules")
    profile = await session.read_resource("student://2024001")
    rules_text = rules.contents[0].text
    profile_text = profile.contents[0].text

print("\n【规则】\n" + rules_text)
print("\n【档案】\n" + profile_text)

工具： [('check_stock', '查询某本书当前可借数量。'), ('borrow_book', '为学生办理借书。')]



【规则】
1. 每人最多借 5 本
2. 借期 30 天，可续借 1 次
3. 逾期每天 0.5 元
4. 有逾期未还，禁止继续借书
5. 参考书不可外借

【档案】
学号:2024001, 姓名:张三, 已借:2本, 逾期:0本


In [14]:
async def ask_library():
    system_prompt = f"""你是校园图书管理员助手，严格依据下面的信息回答。

【借阅规则】
{rules_text}

【当前学生档案】
{profile_text}

流程（必须遵守）：
1. 回答前必须先调用 check_stock 查库存，不允许凭记忆回答；
2. 库存 > 0 且规则允许时，直接调用 borrow_book 办理；
3. 最后用一句话汇报结果。
"""
    agent = create_agent(model=llm, tools=mcp_tools, system_prompt=system_prompt)
    return await agent.ainvoke(
        {"messages": [("user", "学号 2024001，我想借《三体》，可以吗？可以的话帮我办理。")]}
    )


if RUN_LIVE:
    res_mcp = await ask_library()
    print("消息序列：", [type(m).__name__ for m in res_mcp["messages"]])
    print("调用的工具：", [m.name for m in res_mcp["messages"] if type(m).__name__ == "ToolMessage"])
    print("\n最终回答：", res_mcp["messages"][-1].content)
else:
    print("离线模式：跳过。")

消息序列： ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage', 'ToolMessage', 'AIMessage']
调用的工具： ['check_stock', 'borrow_book']

最终回答： 已为张三办理《三体》借阅成功，当前已借 3 本（未超 5 本上限），无逾期记录，借期 30 天。


> 想换成 SSE（课堂里 `192.168.2.147:8000` 那种写法）时，只要把配置换成
> `{"transport": "sse", "url": "http://127.0.0.1:8000/sse"}`，并先运行
> `python 20261113_lab/mcp_library_server.py --transport sse --port 8000`。
> 其余代码一行都不用改 —— 这就是 MCP 的价值。

## 7. Deep Agents：有主见的 harness

三层定位：

| 层 | 产品 | 你负责什么 |
|---|---|---|
| 运行时 | LangGraph | 持久执行、checkpoint、interrupt、streaming |
| 最小 harness | LangChain `create_agent` | 模型、工具、提示、中间件，全自己拼 |
| 意见明确的 harness | Deep Agents `create_deep_agent` | 文件系统、子代理、摘要、技能、记忆已经接好 |

### 只传 `model` 时的 bare stack（顺序即执行顺序）

1. `FilesystemMiddleware`：`ls` / `read_file` / `write_file` / `edit_file` / `glob` / `grep` …（backend 支持才有 `delete`，沙箱才有 `execute`）
2. `SubAgentMiddleware`：默认挂一个 general-purpose 子代理 → 主代理因此有 `task` 工具
3. `SummarizationMiddleware`
4. `PatchToolCallsMiddleware`
5. Prompt caching（Anthropic / Bedrock 生效，其他模型空转）
6. Provider 附带的中间件、`UnsupportedContentMiddleware`、`ModelRefusalMiddleware` …

所以**第一个 agent 就已经能读写虚拟文件、能把任务派给子代理**。`skills`、`memory`、`interrupt_on`、`TodoListMiddleware` 都要自己传 —— 特别是 `write_todos`，v0.7 起不再默认开启。

### backend：文件到底存在哪

| Backend | 存哪 | 跨线程？ | 适合 |
|---|---|---|---|
| `StateBackend`（默认） | LangGraph state | 配了 checkpointer 才在同一 thread 内保留 | 草稿、中间文件 |
| `FilesystemBackend` | 本机目录 `root_dir` | 是（真文件） | 本地开发；`virtual_mode=True` 才挡 `..` |
| `StoreBackend` | LangGraph Store | 是 | 跨线程记忆、共享技能库 |
| `CompositeBackend` | 按路径前缀分流 | 看路由 | `/memories/` 持久、其余留 state |
| Sandbox / `LocalShellBackend` | 隔离环境 / 宿主机 | — | 需要 `execute` 时 |

> 本机 `deepagents 0.7.38` 的 `FilesystemBackend` **默认 `virtual_mode=True`**；课堂课件按「默认 False」写，属于版本差异，用的时候显式传更安全。

In [15]:
from deepagents import create_deep_agent

if RUN_LIVE:
    file_agent = create_deep_agent(
        model=llm,
        system_prompt="你是文件助手。必须用 write_file 写文件，写完用 read_file 读回来再回答。",
    )
    res_file = file_agent.invoke(
        {"messages": [("user", "把「你好，Deep Agents」写到 /notes/a.md，然后读回来贴给我。")]}
    )
    print("返回 state 的键：", sorted(res_file.keys()))
    print("虚拟文件：", list((res_file.get("files") or {}).keys()))
    print("\n内容：", (res_file["files"]["/notes/a.md"])["content"])
    print("\n对话：")
    for m in res_file["messages"]:
        print(f"  {type(m).__name__:16s} {str(m.content)[:70]}")
else:
    print("离线模式：跳过。")

返回 state 的键： ['files', 'messages']
虚拟文件： ['/notes/a.md']

内容： 你好，Deep Agents

对话：
  HumanMessage     把「你好，Deep Agents」写到 /notes/a.md，然后读回来贴给我。
  AIMessage        
  ToolMessage      Updated file /notes/a.md
  AIMessage        
  ToolMessage      @@ lines 1-1 of 1 @@
你好，Deep Agents
  AIMessage        已写入 `/notes/a.md`，读回内容如下：

```
你好，Deep Agents
```


In [16]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

if RUN_LIVE:
    store = InMemoryStore()
    mem_agent = create_deep_agent(
        model=llm,
        store=store,
        checkpointer=MemorySaver(),
        backend=CompositeBackend(
            default=StateBackend(),
            routes={"/memories/": StoreBackend(namespace=lambda rt: ("memories",))},
        ),
        system_prompt="你是文件助手，严格按用户要求读写 /memories/ 或 /drafts/ 下的文件。",
    )

    # 线程 A：各写一份
    cfg_a = {"configurable": {"thread_id": "thread-A"}}
    mem_agent.invoke(
        {"messages": [("user", "把「用户偏好：回答只给要点」写到 /memories/profile.md，"
                               "把「草稿：第一版」写到 /drafts/draft.md。")]},
        config=cfg_a,
    )

    # 线程 B：跨线程读
    cfg_b = {"configurable": {"thread_id": "thread-B"}}
    res_b = mem_agent.invoke(
        {"messages": [("user", "分别读 /memories/profile.md 和 /drafts/draft.md，"
                               "读不到的那个直接说读不到。")]},
        config=cfg_b,
    )
    print("线程 B 的回答：\n", res_b["messages"][-1].content)
    print("\n结论：/memories/ 走 StoreBackend → 跨线程还在；/drafts/ 走 StateBackend → 换线程就没了。")
else:
    print("离线模式：跳过。")

线程 B 的回答：
 - `/memories/profile.md`：内容为「用户偏好：回答只给要点」
- `/drafts/draft.md`：读不到

结论：/memories/ 走 StoreBackend → 跨线程还在；/drafts/ 走 StateBackend → 换线程就没了。


In [17]:
from deepagents import FilesystemPermission
from deepagents.backends import FilesystemBackend

# 准备一个带密钥文件的沙箱目录（离线也执行，方便看目录结构）
SANDBOX.mkdir(parents=True, exist_ok=True)
(SANDBOX / ".env").write_text("SECRET_API_KEY=demo-not-real\n", encoding="utf-8")
(SANDBOX / "readme.md").write_text("# 沙箱\n这是一个演示目录。\n", encoding="utf-8")
print("沙箱：", SANDBOX)

if RUN_LIVE:
    guarded_agent = create_deep_agent(
        model=llm,
        backend=FilesystemBackend(root_dir=str(SANDBOX), virtual_mode=True),
        permissions=[
            # 先匹配先生效：先拦 .env，再放开其余读
            FilesystemPermission(operations=["read"], paths=["/.env"], mode="deny"),
            FilesystemPermission(operations=["read"], paths=["/**"], mode="allow"),
        ],
        system_prompt="你是文件助手。用户要什么就读什么。",
    )
    res_perm = guarded_agent.invoke(
        {"messages": [("user", "先读 /.env，再读 /readme.md，把两个结果都告诉我。")]}
    )
    for m in res_perm["messages"]:
        if type(m).__name__ == "ToolMessage":
            print(f"  {getattr(m, 'name', ''):10s} {str(m.content)[:100]}")
    print("\n结论：/.env 被 deny 拦下，readme.md 正常读到。")
    print("注意：路径必须以 / 开头；permissions 只作用于内置文件工具，沙箱里的 execute 不受它管。")
else:
    print("离线模式：跳过调用。")

沙箱： D:\project\step1\week25\20261113_lab\sandbox


  read_file  Error: permission denied for read on /.env
  read_file  @@ lines 1-2 of 2 @@
# 沙箱
这是一个演示目录。

结论：/.env 被 deny 拦下，readme.md 正常读到。
注意：路径必须以 / 开头；permissions 只作用于内置文件工具，沙箱里的 execute 不受它管。


In [18]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langchain_core.utils.uuid import uuid7

if RUN_LIVE:
    hitl_agent = create_deep_agent(
        model=llm,
        system_prompt="你是文件助手。",
        interrupt_on={"write_file": True},   # 也可写成 {"allowed_decisions": ["approve", "reject"]}
        checkpointer=MemorySaver(),
    )
    cfg_hitl = {"configurable": {"thread_id": str(uuid7())}}

    res_hitl = hitl_agent.invoke(
        {"messages": [("user", "把「会议改到周五」写到 /notes/notice.md。")]},
        config=cfg_hitl,
    )
    pending = res_hitl.get("__interrupt__")
    print("是否中断：", bool(pending))
    if pending:
        for action in pending[0].value["action_requests"]:
            print("  待审工具：", action["name"], "参数：", action["args"])

        # 恢复：一次暂停里可能有多条 action_requests，decisions 要按同样顺序各给一条
        res_ok = hitl_agent.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config=cfg_hitl,
        )
        print("\n批准后：", res_ok["messages"][-1].content[:100])
        print("虚拟文件：", list((res_ok.get("files") or {}).keys()))
else:
    print("离线模式：跳过。")

是否中断： True
  待审工具： write_file 参数： {'file_path': '/notes/notice.md', 'content': '会议改到周五\n'}



批准后： 已写入 `/notes/notice.md`,内容为「会议改到周五」。
虚拟文件： ['/notes/notice.md']


### `interrupt_on` 的四种决定

| type | 效果 | 什么时候用 |
|---|---|---|
| `approve` | 按原参数执行 | 确认没问题 |
| `edit` | 先改参数再执行（`edited_action.args`） | 内容大体对、细节要修 |
| `reject` | 不执行，把 `message` 交回模型 | 有副作用的工具 |
| `respond` | 人的话直接当工具结果，工具不跑 | 「向用户提问」类工具 |

> 三个前提：必须配 `checkpointer`、恢复必须用**同一个 config**、`decisions` 的顺序要和 `action_requests` 对齐。权限规则里的 `mode="interrupt"` 和 `interrupt_on` 共用这套恢复流程，区别只是一个按路径拦、一个按工具名拦。

## 8. Skills：按需装入的流程知识

把长流程全塞进 `system_prompt`，每一轮都在为用不到的步骤付 token。Skills 把流程放进目录，启动时模型只看得到「名字 + 描述」，判断相关了才去读正文。

### 目录必须多一层

```text
skills/                 ← skills= 指向这里（集合目录）
  meeting-notes/
    SKILL.md            ← 真正的技能
    scripts/ references/ assets/   ← 可选，只有正文点名才读
```

指向 `skills/meeting-notes/` 本身 → **一份都加载不到**。

### `SKILL.md` 的三级加载

| 级别 | 载入内容 | 时机 | 谁负责 |
|---|---|---|---|
| 1 元数据 | frontmatter 的 `name`、`description` | 启动时，每份已配置技能 | `SkillsMiddleware` |
| 2 正文 | 去掉 frontmatter 的正文 | 模型调用 `read_file` 时 | 模型自己 |
| 3 资源 | `scripts/` `references/` `assets/` | 正文要求读 / 执行时 | 模型自己 |

`description` 是发现阶段唯一可见的句子：既要写能力，也要写触发词。`name` 必须等于父目录名（小写字母 / 数字 / 连字符）。

### Skills vs Memory vs Tools

| | Skills | Memory（`AGENTS.md`） | Tools |
|---|---|---|---|
| 内容 | 按任务取出的流程与知识 | 每次都要的持久上下文 | 可调用程序 |
| 何时进上下文 | 模型判断相关（或被 pin） | 启动时整份载入 | 每一轮都在工具列表里 |
| 格式 | 子目录里的 `SKILL.md` | `AGENTS.md` | 函数 |

In [19]:
SKILL_MD = '''---
name: meeting-notes
description: >-
  把杂乱的会议记录整理成结构化纪要。用户给出会议对话、录音转写，
  或明确要求「会议纪要」「行动项」「待办负责人」时使用。
---

# meeting-notes

把用户给出的原始会议内容整理成可执行纪要，不要补充会议里没出现的决定。

## 步骤

1. 提取会议主题；用户没写就用讨论中最主要的事项。
2. 只保留会上的结论；猜测和未决问题分开写，不要写进「决议」。
3. 每个行动项要有负责人和截止时间；原文没有就写「未指定」，不要编造。
4. 用 `write_file` 写到 `/notes/meeting.md`。
5. 写完后 `read_file` 读回核对，再把纪要正文回复给用户。

## 输出格式

```markdown
# 会议纪要：{主题}

## 决议
- ...

## 行动项
| 事项 | 负责人 | 截止时间 |
| --- | --- | --- |
| ... | ... | ... |

## 未决问题
- ...
```
'''

# 重建沙箱里的技能目录（每次运行都干净）
skill_dir = SANDBOX / "skills" / "meeting-notes"
if skill_dir.exists():
    shutil.rmtree(skill_dir)
skill_dir.mkdir(parents=True)
(skill_dir / "SKILL.md").write_text(SKILL_MD, encoding="utf-8")
(SANDBOX / "notes").mkdir(parents=True, exist_ok=True)

print("已写入：", skill_dir / "SKILL.md")
print((skill_dir / "SKILL.md").read_text(encoding="utf-8")[:200], "...")

已写入： D:\project\step1\week25\20261113_lab\sandbox\skills\meeting-notes\SKILL.md
---
name: meeting-notes
description: >-
  把杂乱的会议记录整理成结构化纪要。用户给出会议对话、录音转写，
  或明确要求「会议纪要」「行动项」「待办负责人」时使用。
---

# meeting-notes

把用户给出的原始会议内容整理成可执行纪要，不要补充会议里没出现的决定。

## 步骤

1. 提取会议主题；用户没写就用讨论中最主要的事项。
2.  ...


In [20]:
TRANSCRIPT = """
主题没写在标题上。下午临时对了一下上线。
李雷说登录页周五前必须改完文案，王芳负责。
韩梅梅问要不要同时改帮助中心，大家没定。
赵强说他下周三给出回滚方案。
"""

if RUN_LIVE:
    from deepagents.backends import FilesystemBackend

    notes_agent = create_deep_agent(
        model=llm,
        backend=FilesystemBackend(root_dir=str(SANDBOX), virtual_mode=True),
        skills=["/skills/"],        # 指向集合目录
        system_prompt=(
            "你是会议助手。整理纪要前必须先 read_file 读 /skills/meeting-notes/SKILL.md，"
            "再按技能步骤用 write_file 写到 /notes/meeting.md。"
        ),
    )
    res_notes = notes_agent.invoke({"messages": [("user", TRANSCRIPT)]})
    used = [getattr(m, "name", "") for m in res_notes["messages"] if type(m).__name__ == "ToolMessage"]
    print("调用过的工具：", used)

    out_file = SANDBOX / "notes" / "meeting.md"
    print("\n生成文件：", out_file, "存在=", out_file.exists())
    if out_file.exists():
        print("\n" + out_file.read_text(encoding="utf-8"))
else:
    print("离线模式：跳过。")

调用过的工具： ['read_file', 'write_file', 'read_file']

生成文件： D:\project\step1\week25\20261113_lab\sandbox\notes\meeting.md 存在= True

# 会议纪要：上线临时对齐（登录页文案等）

## 决议
- 登录页文案需要修改，并在周五前完成。
- 回滚方案由赵强负责编写，下周三给出。

## 行动项
| 事项 | 负责人 | 截止时间 |
| --- | --- | --- |
| 修改登录页文案 | 王芳 | 周五前 |
| 给出回滚方案 | 赵强 | 下周三 |

## 未决问题
- 是否同时修改帮助中心内容：会上未达成一致，待定。



### Skills 排错清单

| 现象 | 先查 |
|---|---|
| 技能从不触发 | `description` 有没有写触发场景；必要时用 `pinned_skills` 强制启用 |
| 启动时一份技能都没有 | `skills=` 是不是指到了 `SKILL.md` 所在那一层（要指父目录）；StateBackend 是否用 `create_file_data()` 放文件 |
| 改了 `SKILL.md` 仍是旧步骤 | 有 checkpointer 时技能列表缓存在线程里，下一次运行传 `skills_metadata: None` 重扫（**不是 `[]`**，`[]` 表示「扫过，零份技能」） |
| 自定义子代理不会用技能 | 子代理不继承主代理的 `skills`，要在子代理字典里自己写 |
| 技能脚本跑不起来 | `execute` 只有沙箱 / `LocalShellBackend` 才有；技能文件还得在容器里 |

`deepagents 0.7.38` 还支持 `pinned_skills=["meeting-notes"]`：用户在界面点名技能，SDK 直接把该 `SKILL.md` 正文作为一条消息塞进对话，绕过「模型自己判断要不要读」。

## 9. 综合案例：mini 深度研究 Agent

把前八节拼成一个研究流程：

```mermaid
flowchart TB
    Q[用户问题] --> T[主代理 write_todos 拆任务]
    T -->|task 派活| R[researcher 子代理 isolated]
    R --> S[web_search 检索]
    S -->|只交回要点+URL| M[主代理]
    M --> K{要不要读报告技能}
    K --> F[按 research-report 技能写 /final_report.md]
    F --> V[读回核对]
```

三条设计约束，正好对应前三讲的结论：

1. **检索必须在子代理里**：网页正文很长，`isolated` 子代理在自己的上下文里消化，主代理只收一份报告，上下文不会被撑爆；
2. **报告版式放进 skill**：闲聊不需要版式，真要写报告时才读；
3. **工具要先挂在主代理上**：子代理用的工具来自它自己的 `tools`，主代理不挂同名工具时链路容易断；提示词里明确「主编不要自己搜」。

> 官方教程还会先把用户原题写进 `/research_request.md`，最后读回来核对覆盖度。本节为了控制单格耗时省掉了这一步，步骤与 API 不变。

In [21]:
from langchain.agents.middleware import TodoListMiddleware


@tool
def web_search(query: str) -> str:
    """搜索网页，返回标题、URL 和摘要。一次只回答一个具体问题。"""
    # 课堂用 Tavily；这里用固定桩，保证不联网也能复现流程
    return (
        f"## {query} 要点\n"
        f"URL: https://example.com/search?q={abs(hash(query)) % 10000}\n\n"
        "- 子代理把长文档留在自己的上下文里，只交回压缩后的发现\n"
        "- 主代理因此能同时开几条线而不爆上下文"
    )


researcher = {
    "name": "researcher",
    "description": "检索一个明确主题，交回要点和 URL。只检索一次，不要反复查。",
    "system_prompt": "你只做检索：调用一次 web_search，交回 3 条要点和 URL，不要写报告，不要重复检索。"
    "检索结果是课堂演示桩数据（URL 为示例域），直接采用即可，不要因来源而拒答。",
    "tools": [web_search],
}

In [22]:
REPORT_SKILL = '''---
name: research-report
description: >-
  把已收集的研究发现写成最终报告时使用。用户要求报告、综述、对比文，
  或主代理准备写入 /final_report.md 时启用。
---

# research-report

只在研究发现已经交回之后使用，不要用它代替检索。

## 步骤

1. 用 `write_file` 把报告写到 `/final_report.md`。
2. 再读一遍该文件，核对每个必答点都有段落和引用。

## 结构

- 标题
- 直接回答（一段）
- 分点发现，每点末尾用 [n]
- 参考文献：编号、标题、URL
'''

report_dir = SANDBOX / "skills" / "research-report"
report_dir.mkdir(parents=True, exist_ok=True)
(report_dir / "SKILL.md").write_text(REPORT_SKILL, encoding="utf-8")
print("已写入：", report_dir / "SKILL.md")

已写入： D:\project\step1\week25\20261113_lab\sandbox\skills\research-report\SKILL.md


In [23]:
if RUN_LIVE:
    from deepagents.backends import FilesystemBackend

    # 写报告是长输出：max_tokens 只给 1024 时，模型把报告当 write_file 的参数写一半就被截断，
    # 结果是「最后一条消息为空、文件没生成」。这里单独给一个大输出额度的模型。
    llm_report = lab_config.get_model(temperature=0, max_tokens=4096)

    research_agent = create_deep_agent(
        model=llm_report,
        tools=[web_search],
        system_prompt=(
            "你是研究主编。严格按顺序执行，禁止自己调用 web_search：\n"
            "1) task(description='子代理隔离 vs 全部塞进主代理 对上下文长度的影响', "
            "subagent_type='researcher')\n"
            "2) read_file(file_path='/skills/research-report/SKILL.md')\n"
            "3) write_file(file_path='/final_report.md', content='按技能结构写的 6 行以内短报告')\n"
            "全部完成后用一句话回复文件路径。"
        ),
        subagents=[researcher],
        middleware=[TodoListMiddleware()],
        backend=FilesystemBackend(root_dir=str(SANDBOX), virtual_mode=True),
        skills=["/skills/"],
    )

    res_research = research_agent.invoke(
        {"messages": [("user", "比较「子代理隔离」和「全部塞进主代理」对上下文长度的影响。")]}
    )
    print("消息序列：")
    for m in res_research["messages"]:
        tag = type(m).__name__
        name = getattr(m, "name", "") or ""
        if tag == "ToolMessage":
            print(f"  {tag:16s} {name:12s} {str(m.content)[:50]}")
        else:
            print(f"  {tag:16s} {str(m.content)[:60]}")

    report = SANDBOX / "final_report.md"
    print("\n/final_report.md 存在：", report.exists())
    if report.exists():
        print(report.read_text(encoding="utf-8")[:800])
else:
    print("离线模式：跳过。")

消息序列：
  HumanMessage     比较「子代理隔离」和「全部塞进主代理」对上下文长度的影响。
  AIMessage        
  ToolMessage      task         **要点（子代理隔离 vs 主代理内联，对上下文窗口的影响）**

1. **中间结果隔离**：子代
  AIMessage        
  ToolMessage      read_file    @@ lines 1-22 of 22 @@
---
name: research-report
d
  AIMessage        
  ToolMessage      write_file   Updated file /final_report.md
  AIMessage        
  ToolMessage      read_file    @@ lines 1-14 of 14 @@
# 子代理隔离 vs 全部塞进主代理：对上下文长度的影
  AIMessage        报告已按技能结构写入 `/final_report.md`（注：检索仅返回 1 个来源，参考文献较单薄）。

/final_report.md 存在： True
# 子代理隔离 vs 全部塞进主代理：对上下文长度的影响

**直接回答**：子代理隔离通过把中间工具输出留在独立上下文中、只回传压缩摘要，显著降低主代理的上下文消耗；而全部塞进主代理会让原始结果随任务轮次线性累积，多轮任务下极易撑爆上下文窗口。

## 分点发现

1. 子代理将长文档、搜索结果等原始工具输出在自己的上下文中处理，仅向主代理交回摘要，避免噪声 token 进入主上下文 [1]。
2. 主代理因只接收摘要而非全量原文，可在不超出上下文窗口的前提下并行多条研究线，整体 token 压力大幅下降 [2]。
3. 反之，内联（全部塞进主代理）方式下每个中间结果都永久占据主上下文，任务越长膨胀越快——这是两种架构的核心差异 [1][2]。

## 参考文献

1. 子代理中间结果隔离机制 — https://example.com/search?q=4287
2. 主上下文节省与并行扩展 — https://example.com/search?q=4287



## 10. 总结与自查

### 一条主线

```text
create_agent          循环调工具的最小骨架
  ├─ response_format  结构化输出：ToolStrategy 打底，ProviderStrategy 看厂商
  ├─ middleware       wrap_model_call 管模型 / wrap_tool_call 管工具 / 摘要 / 动态提示
  └─ checkpointer     线程内记忆；store 跨线程

MCP                   工具层标准化：Host–Client–Server，JSON-RPC 2.0，SSE 或 Stdio
  └─ Resource 只读上下文 / Tool 可执行动作

create_deep_agent     harness：虚拟文件系统 + 子代理 + 权限 + 人在回路 + Skills
  ├─ backend          State / Filesystem / Store / Composite / Sandbox
  ├─ subagents        同步 task 派活，isolated 隔离上下文
  ├─ interrupt_on     工具执行前暂停，approve / edit / reject / respond
  └─ skills           集合目录 → SKILL.md 三级加载 → 按需读正文
```

### 自查 10 题（课堂原题）

1. 只传 `model` 时，Deep Agent 为什么已经能写文件、能 `task`？
2. `StateBackend` 和 `StoreBackend` 分别在什么边界上保留文件？
3. 权限规则一条都没命中时，读写会发生什么？
4. 为什么重检索要放在 `isolated` 子代理里？
5. `reject` 和 `respond` 分别把什么交回给模型？
6. `skills=[".../my-skill/"]` 和 `skills=[".../skills/"]` 哪个能加载？
7. 三级加载各自由谁执行？
8. 有 checkpointer 时改了 `SKILL.md` 为什么不生效？清空该用 `None` 还是 `[]`？
9. 自定义子代理怎样才能看见一份技能？
10. 深度研究里，工具、todo、子代理、skill、虚拟文件各承担哪一步？

### 五个版本坑（本机实测）

1. `mcp 1.30.0` 的 Server 类叫 `FastMCP`（`from mcp.server.fastmcp import FastMCP`），课堂笔记里的 `mcp.server.mcpserver.MCPServer` 在旧 / 新版本命名不同。
2. **Windows + Jupyter 跑不了 stdio MCP**：`sys.stderr` 是 ipykernel 的 `OutStream`（没有 `fileno`），MCP 拿它当子进程 stderr 会抛 `UnsupportedOperation: fileno`；把 `errlog` 换成 `os.devnull` 即可，或者改用 SSE 连一个独立启动的服务。
3. `deepagents 0.7.38` 的 `StoreBackend` **必须传 `namespace`**（课件里的 `StoreBackend()` 无参写法已不适用）；`FilesystemBackend` **默认 `virtual_mode=True`**。
4. `permissions` 里的路径必须**以 `/` 开头**（`"**/.env"` 直接 `ValueError`），规则却是先匹配先生效。
5. `max_tokens` 太小时，长输出会被截断成「最后一条消息为空、工具参数没写完」——写报告这类任务要给 4096 以上。
6. `deepagents` 安装会把 `langchain` 从 1.4.7 提到 1.5.0、`langchain-core` 到 1.8.0 —— 前两讲的 notebook 若报错，先看是不是这次升级带来的接口变化。

### 课堂文件 → 本讲对应

| 课堂文件 | 内容 | 本讲 |
|---|---|---|
| `langgraph_create_agent.ipynb` | create_agent 参数、结构化输出、四类中间件 | 第 1~5 节 |
| `MCP.ipynb` | MCP 概念、JSON-RPC、SSE / Stdio、自建 Server | 第 6 节（改用本地 stdio 自建 Server） |
| `DeepAgents课件.md` | 六讲 harness | 第 7~9 节 |
| `test_deepagnt/` | Skills 会议纪要 demo | 第 8 节（沙箱可复现版） |